# Assignment 1 — QANet (Direct Rerun Notebook)

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

This notebook is organized so you can restart the runtime and rerun it from top to bottom.

> Before training, switch Colab to **GPU runtime**.
> The notebook now includes automatic preflight checks and will stop early if the repo is stale or the data/labels/model sanity checks fail.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Assignment1_2026_fixed"


In [ ]:
# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en_core_web_sm


---
## Section 0 — Environment Setup

Mount Google Drive, install dependencies, set the working directory, and then force-sync the repo so Colab does not keep using an older checkout or cached module state.


In [ ]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Working directory:', os.getcwd())
print('sys.path[0]:', sys.path[0])


---
## Section 0.5 — Force-Sync the Repo and Reset Python Module State

> Run this after mounting Drive and setting `PROJECT_ROOT`. It updates the repo, verifies required files exist, and clears stale Python module cache so the rest of the notebook uses the latest code.


In [ ]:
import os, sys
from importlib import invalidate_caches

!git -C {PROJECT_ROOT} pull origin main
!git -C {PROJECT_ROOT} rev-parse --short HEAD
!ls {PROJECT_ROOT}/Tools

required_files = [
    os.path.join(PROJECT_ROOT, 'Tools', 'data_diagnostics.py'),
    os.path.join(PROJECT_ROOT, 'TrainTools', 'overfit_check.py'),
]
for path in required_files:
    assert os.path.exists(path), f'Missing required file after git pull: {path}'

for prefix in ['Tools', 'TrainTools', 'EvaluateTools', 'Models', 'Data', 'Losses', 'Optimizers', 'Schedulers']:
    for name in list(sys.modules.keys()):
        if name == prefix or name.startswith(prefix + '.'):
            del sys.modules[name]

invalidate_caches()
print('Repo sync complete and stale module cache cleared.')


> If this cell fails, do **not** continue. Restart the runtime and rerun from the top until the required files are visible.


---
## Section 1 — Download Full Data *(one-time setup)*

Downloads the full SQuAD v1.1 train/dev set and the full GloVe 840B 300d vectors into `_data/`.

> This is the recommended setup if your goal is to improve EM substantially. The mini dataset is useful only for pipeline smoke tests.


In [ ]:
from Tools.download import download

download(data_dir="_data")


---
## Section 2 — Preprocess Full Data *(rebuild from scratch for a real retrain)*

Tokenises the full SQuAD JSON files, builds word/char vocabularies from the full GloVe file, and writes padded index tensors to `_data/`.

> If you updated the repo code, you should **delete the old preprocessing outputs and rebuild them** before training again. This is especially important after embedding / vocab fixes.


In [ ]:
from pathlib import Path
import shutil
from Tools.preproc import preprocess

for path in [
    '_data/train.npz', '_data/dev.npz',
    '_data/word_emb.json', '_data/char_emb.json',
    '_data/word2idx.json', '_data/char2idx.json',
    '_data/train_eval.json', '_data/dev_eval.json', '_data/dev_meta.json',
]:
    p = Path(path)
    if p.exists():
        p.unlink()

preprocess(
    train_file='_data/squad/train-v1.1.json',
    dev_file='_data/squad/dev-v1.1.json',
    glove_word_file='_data/glove/glove.840B.300d.txt',
    target_dir='_data',
    para_limit=400,
    ques_limit=50,
)


---
## Section 2.2 — Inspect Preprocessing Statistics

> This confirms that rebuilt `_data/` outputs were generated with the latest preprocessing logic.


In [ ]:
import json

with open('_data/preprocess_stats.json', 'r') as f:
    preprocess_stats = json.load(f)

print(json.dumps(preprocess_stats, indent=2))

assert preprocess_stats['train_record_count'] > 80000, 'Unexpectedly low train_record_count'
assert preprocess_stats['dev_record_count'] > 10000, 'Unexpectedly low dev_record_count'
assert preprocess_stats['word_embedding']['pretrained_hits'] > 80000, 'Pretrained word embedding coverage looks too low'
print('Preprocess stats sanity check passed.')


---
## Section 2.25 — Check Label Upper Bound

> This decodes the stored gold `(y1, y2)` spans back into text and compares them against the official answers. If this upper bound is low, the preprocessing labels are misaligned and long training will fail.


In [ ]:
from importlib import invalidate_caches
invalidate_caches()

from Tools.data_diagnostics import label_upper_bound

train_label_upper = label_upper_bound('_data/train.npz', '_data/train_eval.json', limit=2000)
dev_label_upper = label_upper_bound('_data/dev.npz', '_data/dev_eval.json', limit=2000)

print('TRAIN LABEL UPPER:', train_label_upper)
print('DEV LABEL UPPER:', dev_label_upper)

assert train_label_upper['f1'] > 80, f"Train label upper bound too low: {train_label_upper}"
assert dev_label_upper['f1'] > 80, f"Dev label upper bound too low: {dev_label_upper}"
print('Label upper-bound sanity check passed.')


---
## Section 2.3 — Tiny-Subset Overfit Check

> The previous overfit check could still be too hard on a real subset. This version makes the task intentionally easier: smaller subset, more steps, no dropout, constant LR. If this still fails, there is almost certainly a remaining model bug.


In [ ]:
from TrainTools.overfit_check import overfit_check

overfit_results = overfit_check(
    train_npz='_data/train.npz',
    word_emb_json='_data/word_emb.json',
    char_emb_json='_data/char_emb.json',
    train_eval_json='_data/train_eval.json',
    subset_size=16,
    save_dir='_overfit_check',
    log_dir='_overfit_log',
    num_steps=1000,
    checkpoint=200,
    batch_size=8,
    learning_rate=5e-3,
    dropout=0.0,
    dropout_char=0.0,
    scheduler_name='lambda',
)

print(overfit_results['eval_metrics'])

assert overfit_results['eval_metrics']['f1'] > 80, f"Overfit check failed: {overfit_results['eval_metrics']}"
assert overfit_results['eval_metrics']['exact_match'] > 60, f"Overfit check EM too low: {overfit_results['eval_metrics']}"
print('Tiny-subset overfit sanity check passed.')


---
## Section 2.5 — Clear Old Checkpoints Before a Fresh Full Retrain

> Run this once before starting a new full-data training run, so you do not accidentally evaluate stale checkpoints.


In [ ]:
from pathlib import Path
import shutil

for path in ['_model_full', '_log_full']:
    p = Path(path)
    if p.exists():
        shutil.rmtree(p)
        print(f'removed {p}')

Path('_model_full').mkdir(exist_ok=True)
Path('_log_full').mkdir(exist_ok=True)
print('fresh output directories ready')


---
## Section 3 — Long Full-Data Training

This cell should only be run after all preflight checks above have passed. It trains QANet on full SQuAD v1.1 and saves the best checkpoint by dev EM/F1 to `_model_full/model.pt`.

> For the current high-memory Colab setup, start with `batch_size=32`, `grad_accum_steps=1`, and `num_steps=20000`. If this unexpectedly runs out of memory, fall back to `batch_size=16`.


In [ ]:
from TrainTools.train import train

results = train(
    train_npz       = '_data/train.npz',
    dev_npz         = '_data/dev.npz',
    word_emb_json   = '_data/word_emb.json',
    char_emb_json   = '_data/char_emb.json',
    train_eval_json = '_data/train_eval.json',
    dev_eval_json   = '_data/dev_eval.json',
    save_dir        = '_model_full',
    log_dir         = '_log_full',
    num_steps       = 20000,
    checkpoint      = 2000,
    batch_size      = 32,
    seed            = 42,
    grad_accum_steps = 1,
    optimizer_name  = 'adam',
    scheduler_name  = 'cosine',
    learning_rate   = 1e-3,
    loss_name       = 'qa_nll',
    norm_name       = 'layer_norm',
    test_num_batches = -1,
    max_answer_len  = 30,
)

print(f"Best F1: {results['best_f1']:.3f}%  |  Best EM: {results['best_em']:.3f}%")


---
## Section 4 — Evaluate Best Checkpoint

Loads the saved best checkpoint and runs inference on the **full dev set**.


In [ ]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz          = '_data/dev.npz',
    word_emb_json    = '_data/word_emb.json',
    char_emb_json    = '_data/char_emb.json',
    dev_eval_json    = '_data/dev_eval.json',
    save_dir         = '_model_full',
    log_dir          = '_log_full',
    ckpt_name        = 'model.pt',
    test_num_batches = -1,
    max_answer_len   = 30,
)

print(f"F1: {metrics['f1']:.3f}%  |  EM: {metrics['exact_match']:.3f}%  |  Loss: {metrics['loss']:.6f}")
